# AgentCore Gateway Metrics - Comprehensive Analysis

## Overview

This notebook provides exhaustive monitoring of **AWS Bedrock AgentCore Gateway** metrics. After completing Lab 3, your AgentCore Gateway resource automatically emits detailed metrics to Amazon CloudWatch for all tool invocations and MCP (Model Context Protocol) operations.

## What You'll Learn

- ✅ **CloudWatch Concepts**: Dimensions, Operations, Target Types for Gateway
- ✅ **MCP Operations**: All 4+ MCP operations including CallToolMcp, InitializeMcp
- ✅ **Console Navigation**: Step-by-step CloudWatch console access
- ✅ **Comprehensive Querying**: Query ALL target types and operations
- ✅ **Performance Monitoring**: Tool execution time analysis

## Prerequisites

- ✅ Completed Lab 3 (AgentCore Gateway)
- ✅ AgentCore Gateway resource deployed
- ✅ Some tool usage (Lambda function calls, MCP tools)
- ✅ AWS CloudWatch access permissions

## 📚 CloudWatch Concepts for AgentCore Gateway

Before diving into metrics, let's understand Gateway-specific concepts:

### 🏷️ **Gateway Dimensions**
Gateway metrics use different dimensions than Memory:

```python
# Examples of Gateway dimensions:
{
    'Resource': 'arn:aws:bedrock-agentcore:us-east-1:123456789012:gateway/my-gateway-abc123',
    'Operation': 'CallToolMcp'  # MCP operation being performed
}

{
    'Resource': 'arn:aws:bedrock-agentcore:us-east-1:123456789012:gateway/my-gateway-abc123',
    'TargetType': 'LAMBDA',  # Type of target being invoked
    'TargetName': 'OrderLookupFunction'  # Specific function/tool name
}
```

### ⚙️ **Gateway Operations** 
Gateway operations are **MCP (Model Context Protocol)** specific:

| Operation | Description | When It Happens |
|-----------|-------------|----------------|
| `CallToolMcp` | Execute a tool through MCP | Every tool invocation |
| `InitializeMcp` | Initialize MCP connection | Gateway startup/connection |
| `InitializedNotificationMcp` | MCP initialization complete | After successful MCP setup |
| `ListToolsMcp` | List available MCP tools | Tool discovery phase |
| `InvokeTarget` | Direct target invocation | Lambda/OpenAPI calls |
| `ListTargets` | List available targets | Target discovery |

### 🎯 **Target Types**
Gateway can invoke different types of targets:

| Target Type | Description | Example |
|-------------|-------------|----------|
| `LAMBDA` | AWS Lambda functions | Order lookup, customer service |
| `OpenAPI` | REST API endpoints | External service calls |
| `MCP` | Model Context Protocol tools | Specialized agent tools |

### 📊 **Gateway Metrics**
Gateway metrics focus on **tool performance and execution**:

| Metric Name | Unit | Description |
|-------------|------|-------------|
| `Invocations` | Count | Total requests to Data Plane API |
| `Latency` | Milliseconds | Time from request to first response token |
| `Duration` | Milliseconds | Complete end-to-end processing time |
| `TargetExecutionTime` | Milliseconds | Time taken by Lambda/OpenAPI targets |
| `TargetType.LAMBDA` | Count | Number of Lambda target invocations |
| `TargetType.OpenAPI` | Count | Number of OpenAPI target invocations |
| `TargetType.MCP` | Count | Number of MCP target invocations |
| `SystemErrors` | Count | AWS server-side errors (5xx) |
| `UserErrors` | Count | Client-side errors (4xx except 429) |
| `Throttles` | Count | Rate-limited requests (429) |

### 🏠 **Namespace**
All AgentCore Gateway metrics live in: **`AWS/Bedrock-AgentCore`**

### 🔄 **Key Differences from Memory Metrics**
- **Performance Focus**: Gateway metrics emphasize tool execution performance
- **Target-Centric**: Metrics track different target types (Lambda, OpenAPI, MCP)
- **Real-Time**: Gateway metrics reflect immediate tool usage patterns
- **Execution Time**: Separate metrics for gateway processing vs target execution

## Step 1: Setup and Resource Discovery

In [ ]:
import boto3
import json
import pandas as pd
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, HTML

# Import utilities for getting lab resources
from scripts.utils import get_ssm_parameter

# Initialize AWS clients
session = boto3.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
cloudwatch = boto3.client('cloudwatch', region_name=region)

print(f"✅ Connected to AWS Account: {account_id}")
print(f"📍 Region: {region}")
print(f"📊 CloudWatch Namespace: AWS/Bedrock-AgentCore")
print(f"🎯 Focus: Gateway Tool Performance & MCP Operations")

# Configure plotting
plt.style.use('default')
sns.set_palette("Set2")
%matplotlib inline

In [ ]:
def get_gateway_resource() -> Dict[str, str]:
    """Get Gateway resource details from Lab 3"""
    
    try:
        gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
        gateway_arn = f"arn:aws:bedrock-agentcore:{region}:{account_id}:gateway/{gateway_id}"
        
        print(f"✅ Found AgentCore Gateway Resource:")
        print(f"   🆔 Gateway ID: {gateway_id}")
        print(f"   🔗 Gateway ARN: {gateway_arn}")
        print(f"   🎯 Monitors: Tool invocations, MCP operations, target performance")
        
        return {
            'gateway_id': gateway_id,
            'gateway_arn': gateway_arn
        }
        
    except Exception as e:
        print(f"❌ Gateway resource not found: {str(e)}")
        print(f"   Please complete Lab 3 first to create AgentCore Gateway")
        return {}

# Get gateway resource
gateway_resource = get_gateway_resource()

## Step 2: Discover ALL Available Gateway Metrics

Let's discover what metrics, operations, and target types are available for your gateway:

In [ ]:
def discover_gateway_metrics_complete(gateway_arn: str) -> Dict:
    """Discover ALL available metrics, operations, and target types for Gateway"""
    
    if not gateway_arn:
        print("⚠️ No Gateway ARN available")
        return {}
    
    print(f"🔍 Discovering ALL Gateway Metrics and Operations")
    print(f"🔗 Resource: {gateway_arn}")
    print("-" * 70)
    
    discovered = {
        'metrics': {},
        'operations': set(),
        'dimensions': set(),
        'target_types': set(),
        'target_names': set(),
        'names': set()  # For agent/tool names
    }
    
    try:
        # Find all metrics for this gateway resource
        response = cloudwatch.list_metrics(
            Namespace='AWS/Bedrock-AgentCore',
            Dimensions=[
                {'Name': 'Resource', 'Value': gateway_arn}
            ]
        )
        
        print(f"✅ Found {len(response['Metrics'])} metric combinations")
        print()
        
        # Analyze each metric
        for metric in response['Metrics']:
            metric_name = metric['MetricName']
            
            if metric_name not in discovered['metrics']:
                discovered['metrics'][metric_name] = []
            
            # Extract dimension information
            dim_combo = {}
            for dim in metric['Dimensions']:
                dim_combo[dim['Name']] = dim['Value']
                discovered['dimensions'].add(dim['Name'])
                
                # Collect specific dimension values
                if dim['Name'] == 'Operation':
                    discovered['operations'].add(dim['Value'])
                elif dim['Name'] == 'TargetType':
                    discovered['target_types'].add(dim['Value'])
                elif dim['Name'] == 'TargetName':
                    discovered['target_names'].add(dim['Value'])
                elif dim['Name'] == 'Name':
                    discovered['names'].add(dim['Value'])
            
            discovered['metrics'][metric_name].append(dim_combo)
        
        # Display comprehensive summary
        print(f"📈 **Available Metrics**: {', '.join(sorted(discovered['metrics'].keys()))}")
        print(f"📐 **Dimensions Used**: {', '.join(sorted(discovered['dimensions']))}")
        
        if discovered['operations']:
            print(f"\n⚙️ **Gateway Operations Found** ({len(discovered['operations'])}):**")
            for i, op in enumerate(sorted(discovered['operations']), 1):
                # Provide context for each operation
                context = ""
                if 'Mcp' in op:
                    context = " (MCP Protocol)"
                elif 'Target' in op:
                    context = " (Target Management)"
                print(f"   {i}. **{op}**{context}")
        
        if discovered['target_types']:
            print(f"\n🎯 **Target Types**: {', '.join(sorted(discovered['target_types']))}")
            
        if discovered['target_names']:
            print(f"🏷️ **Target Names**: {', '.join(sorted(discovered['target_names']))}")
            
        if discovered['names']:
            print(f"📛 **Agent/Tool Names**: {', '.join(sorted(discovered['names']))}")
        
        # Show detailed metric breakdown
        print(f"\n📊 **Detailed Metric Configurations**:")
        for metric_name, combinations in discovered['metrics'].items():
            print(f"\n**{metric_name}** ({len(combinations)} combinations):")
            for combo in combinations[:5]:  # Show first 5
                dims = []
                for k, v in combo.items():
                    if k != 'Resource':
                        display_value = v.split('/')[-1] if '/' in v else v
                        dims.append(f"{k}={display_value}")
                dim_str = ', '.join(dims) if dims else 'Base metric'
                print(f"   • {dim_str}")
            if len(combinations) > 5:
                print(f"   ... and {len(combinations) - 5} more")
        
        # MCP Operation Analysis
        mcp_operations = [op for op in discovered['operations'] if 'Mcp' in op]
        if mcp_operations:
            print(f"\n🔄 **MCP Operations Analysis**:")
            print(f"   Found {len(mcp_operations)} MCP operations:")
            for op in sorted(mcp_operations):
                if op == 'CallToolMcp':
                    print(f"   • **{op}**: 🔧 Tool execution calls")
                elif op == 'InitializeMcp':
                    print(f"   • **{op}**: 🚀 MCP connection setup")
                elif op == 'InitializedNotificationMcp':
                    print(f"   • **{op}**: ✅ MCP ready notification")
                elif op == 'ListToolsMcp':
                    print(f"   • **{op}**: 📋 Tool discovery")
                else:
                    print(f"   • **{op}**: Custom MCP operation")
    
    except Exception as e:
        print(f"❌ Error discovering metrics: {str(e)}")
    
    return discovered

# Discover all gateway metrics
if gateway_resource:
    gateway_discovery = discover_gateway_metrics_complete(gateway_resource['gateway_arn'])
else:
    gateway_discovery = {}

## Step 3: Query Gateway Metrics Exhaustively

Now let's query ALL metrics with their proper dimension combinations:

In [ ]:
def query_gateway_metrics_exhaustive(gateway_arn: str, gateway_id: str, hours_back: int = 24) -> Dict:
    """Query Gateway metrics with ALL possible dimension combinations"""
    
    if not gateway_arn:
        print("⚠️ No Gateway ARN available")
        return {}
    
    print(f"📊 Querying ALL Gateway Metrics (Last {hours_back} hours)")
    print(f"🔗 Resource: {gateway_arn}")
    print(f"🆔 Gateway ID: {gateway_id}")
    print("-" * 70)
    
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=hours_back)
    
    # All possible dimension combinations based on AWS documentation and discovery
    dimension_sets = [
        # 1. Resource-only (base metrics)
        [{'Name': 'Resource', 'Value': gateway_arn}],
        
        # 2. Resource + Operation (MCP operations)
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'Operation', 'Value': 'CallToolMcp'}],
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'Operation', 'Value': 'InitializeMcp'}],
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'Operation', 'Value': 'InitializedNotificationMcp'}],
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'Operation', 'Value': 'ListToolsMcp'}],
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'Operation', 'Value': 'InvokeTarget'}],
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'Operation', 'Value': 'ListTargets'}],
        
        # 3. Resource + TargetType
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'TargetType', 'Value': 'LAMBDA'}],
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'TargetType', 'Value': 'OpenAPI'}],
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'TargetType', 'Value': 'MCP'}],
        
        # 4. Just Operation (global metrics)
        [{'Name': 'Operation', 'Value': 'CallToolMcp'}],
        [{'Name': 'Operation', 'Value': 'InitializeMcp'}],
        [{'Name': 'Operation', 'Value': 'ListToolsMcp'}],
        
        # 5. Combinations with discovered names (if any)
        [{'Name': 'Resource', 'Value': gateway_arn}, {'Name': 'Operation', 'Value': 'CallToolMcp'}, {'Name': 'Name', 'Value': 'default'}],
    ]
    
    # All metrics to query based on AWS documentation
    metrics_to_query = [
        ('Invocations', 'Sum', 'Total requests to Data Plane API'),
        ('Latency', 'Average', 'Time from request to first response token (ms)'),
        ('Duration', 'Average', 'Complete end-to-end processing time (ms)'),
        ('TargetExecutionTime', 'Average', 'Time taken by Lambda/OpenAPI targets (ms)'),
        ('TargetType.LAMBDA', 'Sum', 'Number of Lambda target invocations'),
        ('TargetType.OpenAPI', 'Sum', 'Number of OpenAPI target invocations'),
        ('TargetType.MCP', 'Sum', 'Number of MCP target invocations'),
        ('SystemErrors', 'Sum', 'AWS server-side errors (5xx)'),
        ('UserErrors', 'Sum', 'Client-side errors (4xx except 429)'),
        ('Throttles', 'Sum', 'Rate-limited requests (429)'),
        ('Errors', 'Sum', 'Total errors during requests')
    ]
    
    results = {}
    total_found = 0
    
    for i, dimensions in enumerate(dimension_sets, 1):
        # Create readable dimension string
        dim_parts = []
        for d in dimensions:
            if d['Name'] == 'Resource':
                dim_parts.append(f"Resource={gateway_id}")
            else:
                dim_parts.append(f"{d['Name']}={d['Value']}")
        dim_str = ', '.join(dim_parts)
        
        print(f"\n🔍 **Query {i}/{len(dimension_sets)}**: {dim_str}")
        
        found_metrics = []
        for metric_name, statistic, description in metrics_to_query:
            try:
                response = cloudwatch.get_metric_statistics(
                    Namespace='AWS/Bedrock-AgentCore',
                    MetricName=metric_name,
                    Dimensions=dimensions,
                    StartTime=start_time,
                    EndTime=end_time,
                    Period=3600,  # 1 hour periods
                    Statistics=[statistic]
                )
                
                datapoints = response.get('Datapoints', [])
                if datapoints:
                    sorted_points = sorted(datapoints, key=lambda x: x['Timestamp'])
                    latest = sorted_points[-1][statistic]
                    
                    found_metrics.append({
                        'metric': metric_name,
                        'value': latest,
                        'description': description,
                        'datapoints': len(datapoints)
                    })
                    
                    # Store in results
                    key = f"{metric_name}_{dim_str}"
                    results[key] = {
                        'metric_name': metric_name,
                        'dimensions': dimensions,
                        'datapoints': sorted_points,
                        'latest_value': latest,
                        'description': description
                    }
                    
            except Exception as e:
                continue  # Skip failed queries
        
        if found_metrics:
            for metric_info in found_metrics:
                print(f"   ✅ **{metric_info['metric']}**: {metric_info['value']:.2f} ({metric_info['datapoints']} data points)")
                print(f"      {metric_info['description']}")
            total_found += len(found_metrics)
        else:
            print(f"   ℹ️ No data found")
    
    print(f"\n🎉 **Summary**: Found {total_found} metrics across {len(dimension_sets)} dimension combinations")
    
    # Analyze MCP vs Target performance
    if results:
        print(f"\n🔄 **MCP Operations Analysis**:")
        mcp_invocations = 0
        lambda_invocations = 0
        
        for key, data in results.items():
            if data['metric_name'] == 'Invocations':
                for dim in data['dimensions']:
                    if dim['Name'] == 'Operation' and 'Mcp' in dim['Value']:
                        mcp_invocations += data['latest_value']
                    elif dim['Name'] == 'TargetType' and dim['Value'] == 'LAMBDA':
                        lambda_invocations += data['latest_value']
        
        if mcp_invocations > 0:
            print(f"   • MCP Operations: {mcp_invocations:.0f} invocations")
        if lambda_invocations > 0:
            print(f"   • Lambda Targets: {lambda_invocations:.0f} invocations")
    
    return results

# Query all gateway metrics
if gateway_resource:
    gateway_metrics = query_gateway_metrics_exhaustive(
        gateway_resource['gateway_arn'], 
        gateway_resource['gateway_id']
    )
else:
    gateway_metrics = {}

## Step 4: 🖥️ CloudWatch Console Navigation Guide

Here's how to access Gateway metrics in the AWS CloudWatch console:

In [ ]:
def generate_gateway_console_navigation_guide(gateway_id: str, region: str) -> None:
    """Generate CloudWatch console navigation URLs and instructions for Gateway"""
    
    if not gateway_id:
        print("⚠️ No Gateway ID available for console navigation")
        return
    
    print(f"🖥️ **Gateway CloudWatch Console Navigation Guide**")
    print(f"📍 Region: {region}")
    print(f"🆔 Gateway ID: {gateway_id}")
    print(f"🎯 Focus: Tool Performance & MCP Operations")
    print("-" * 70)
    
    # Base CloudWatch metrics URL
    base_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}"
    
    print(f"### 📊 **Method 1: Direct Gateway Metric URLs**")
    print()
    
    # Specific URLs for Gateway views
    urls = {
        "All Gateway Metrics": f"{base_url}#metricsV2?graph=~()&query=~'*7bAWS*2fBedrock-AgentCore*2cResource*7d&search={gateway_id}",
        "MCP Operations Only": f"{base_url}#metricsV2?graph=~()&query=~'*7bAWS*2fBedrock-AgentCore*2cResource*2cOperation*7d&search=Mcp",
        "Tool Performance": f"{base_url}#metricsV2?graph=~()&query=~'*7bAWS*2fBedrock-AgentCore*2cResource*2cOperation*7d&search=CallToolMcp",
        "Target Type Analysis": f"{base_url}#metricsV2?graph=~()&query=~'*7bAWS*2fBedrock-AgentCore*2cResource*2cTargetType*7d&search={gateway_id}"
    }
    
    for name, url in urls.items():
        print(f"**{name}:**")
        print(f"```")
        print(f"{url}")
        print(f"```")
        print()
    
    print(f"### 📋 **Method 2: Manual Navigation Steps**")
    print()
    print(f"1. **Open CloudWatch Console**:")
    print(f"   - Go to AWS Console → CloudWatch")
    print(f"   - Or use: {base_url}")
    print()
    print(f"2. **Navigate to Gateway Metrics**:")
    print(f"   - Left sidebar → 'Metrics' → 'All metrics'")
    print(f"   - Search: `AWS/Bedrock-AgentCore`")
    print(f"   - Filter by: `{gateway_id}`")
    print()
    print(f"3. **View MCP Operations**:")
    print(f"   - Select 'Resource, Operation' dimension")
    print(f"   - Look for MCP operations:")
    
    mcp_operations = ['CallToolMcp', 'InitializeMcp', 'InitializedNotificationMcp', 'ListToolsMcp']
    for op in mcp_operations:
        icon = "🔧" if op == 'CallToolMcp' else "🚀" if op == 'InitializeMcp' else "✅" if 'Notification' in op else "📋"
        print(f"     {icon} {op}")
    
    print()
    print(f"4. **View Target Performance**:")
    print(f"   - Select 'Resource, TargetType' dimension")
    print(f"   - Monitor target types:")
    print(f"     🔹 LAMBDA - AWS Lambda function calls")
    print(f"     🔹 OpenAPI - REST API endpoint calls")
    print(f"     🔹 MCP - Model Context Protocol tools")
    
    print()
    print(f"### 🎛️ **Method 3: Performance Analysis Searches**")
    print()
    print(f"**Gateway-Specific Search Queries:**")
    print()
    
    searches = [
        ("Tool Execution Performance", f'AWS/Bedrock-AgentCore TargetExecutionTime {gateway_id}'),
        ("MCP Operation Latency", f'AWS/Bedrock-AgentCore Latency CallToolMcp {gateway_id}'),
        ("Gateway vs Target Time", f'AWS/Bedrock-AgentCore Duration Latency TargetExecutionTime {gateway_id}'),
        ("Tool Usage Patterns", f'AWS/Bedrock-AgentCore Invocations Operation {gateway_id}'),
        ("Error Analysis", f'AWS/Bedrock-AgentCore SystemErrors UserErrors Throttles {gateway_id}')
    ]
    
    for name, search in searches:
        print(f"**{name}:**")
        print(f"```")
        print(f"{search}")
        print(f"```")
        print()
    
    print(f"### 📈 **Method 4: Custom Dashboard Creation**")
    print()
    print(f"**Recommended Dashboard Widgets:**")
    print()
    print(f"1. **Tool Performance Overview**:")
    print(f"   - Metric: `Invocations` by `Operation`")
    print(f"   - Shows: Which tools are used most")
    print()
    print(f"2. **Execution Time Analysis**:")
    print(f"   - Metrics: `TargetExecutionTime`, `Latency`, `Duration`")
    print(f"   - Shows: Gateway overhead vs tool execution time")
    print()
    print(f"3. **Target Type Distribution**:")
    print(f"   - Metrics: `TargetType.LAMBDA`, `TargetType.OpenAPI`")
    print(f"   - Shows: Usage patterns across different target types")
    print()
    print(f"4. **Error Monitoring**:")
    print(f"   - Metrics: `SystemErrors`, `UserErrors`, `Throttles`")
    print(f"   - Shows: Gateway reliability and performance issues")
    
    print(f"\n### 🎯 **Gateway-Specific Insights**")
    print()
    print(f"**Key Metrics to Watch:**")
    print(f"   • **TargetExecutionTime**: How long your Lambda functions take")
    print(f"   • **Latency vs Duration**: Gateway processing overhead")
    print(f"   • **CallToolMcp Invocations**: Most important for tool usage")
    print(f"   • **InitializeMcp**: Gateway startup and connection health")
    print()
    print(f"**Performance Optimization:**")
    print(f"   • High TargetExecutionTime → Optimize Lambda functions")
    print(f"   • High Latency → Check network or gateway configuration")
    print(f"   • High UserErrors → Review tool calling patterns")

# Generate console navigation guide
if gateway_resource:
    generate_gateway_console_navigation_guide(gateway_resource['gateway_id'], region)

## Step 5: Visualize Gateway Metrics

Let's create charts to visualize the gateway performance and tool usage patterns:

In [ ]:
def visualize_gateway_metrics(metrics_data: Dict) -> None:
    """Create comprehensive visualizations of gateway metrics"""
    
    if not metrics_data:
        print("📊 No metrics data available for visualization")
        print("   Ensure your AgentCore Gateway has been used recently (tool calls in last 24 hours)")
        return
    
    print(f"📊 **Gateway Metrics Visualization**")
    print(f"📈 Creating charts for {len(metrics_data)} metrics...")
    
    # Organize metrics by category
    performance_metrics = []
    usage_metrics = []
    error_metrics = []
    target_metrics = []
    
    for key, data in metrics_data.items():
        metric_name = data['metric_name']
        if metric_name in ['Latency', 'Duration', 'TargetExecutionTime']:
            performance_metrics.append((key, data))
        elif metric_name in ['Invocations'] or metric_name.startswith('TargetType.'):
            if metric_name.startswith('TargetType.'):
                target_metrics.append((key, data))
            else:
                usage_metrics.append((key, data))
        elif metric_name in ['Errors', 'SystemErrors', 'UserErrors', 'Throttles']:
            error_metrics.append((key, data))
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('AgentCore Gateway Metrics Dashboard', fontsize=16, fontweight='bold')
    
    # Performance Metrics (Latency, Duration, TargetExecutionTime)
    if performance_metrics:
        ax = axes[0, 0]
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
        for i, (key, data) in enumerate(performance_metrics[:4]):
            timestamps = [dp['Timestamp'] for dp in data['datapoints']]
            values = [dp.get('Average', dp.get('Sum', 0)) for dp in data['datapoints']]
            
            label = data['metric_name']
            if data['metric_name'] == 'TargetExecutionTime':
                label += ' (Tool Time)'
            elif data['metric_name'] == 'Latency':
                label += ' (Gateway)'
            
            ax.plot(timestamps, values, marker='o', label=label, color=colors[i % len(colors)])
        
        ax.set_title('🚀 Gateway Performance Metrics', fontweight='bold')
        ax.set_ylabel('Time (milliseconds)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
    else:
        axes[0, 0].text(0.5, 0.5, 'No Performance Data\n🔧 Execute tools to see metrics', 
                       ha='center', va='center', fontsize=12)
        axes[0, 0].set_title('🚀 Gateway Performance Metrics')
    
    # MCP Operations Usage
    if usage_metrics:
        ax = axes[0, 1]
        
        # Group by operation type
        mcp_operations = {}
        for key, data in usage_metrics:
            if data['metric_name'] == 'Invocations':
                for dim in data['dimensions']:
                    if dim['Name'] == 'Operation' and 'Mcp' in dim['Value']:
                        op_name = dim['Value'].replace('Mcp', '')
                        total_invocations = sum(dp.get('Sum', 0) for dp in data['datapoints'])
                        mcp_operations[op_name] = mcp_operations.get(op_name, 0) + total_invocations
        
        if mcp_operations:
            # Create pie chart for MCP operations
            wedges, texts, autotexts = ax.pie(mcp_operations.values(), labels=mcp_operations.keys(), 
                                             autopct='%1.1f%%', startangle=90)
            # Enhance text readability
            for autotext in autotexts:
                autotext.set_color('white')
                autotext.set_fontweight('bold')
            ax.set_title('⚙️ MCP Operations Distribution', fontweight='bold')
        else:
            # Show bar chart of invocations over time
            for key, data in usage_metrics[:3]:
                values = [dp.get('Sum', 0) for dp in data['datapoints']]
                ax.bar(range(len(values)), values, alpha=0.7, 
                      label=data['metric_name'])
            ax.set_title('⚙️ Gateway Usage Metrics', fontweight='bold')
            ax.set_ylabel('Count')
            ax.legend()
            ax.grid(True, alpha=0.3)
    else:
        axes[0, 1].text(0.5, 0.5, 'No Usage Data\n🔧 Make tool calls to see usage', 
                       ha='center', va='center', fontsize=12)
        axes[0, 1].set_title('⚙️ MCP Operations Distribution')
    
    # Target Type Performance
    if target_metrics:
        ax = axes[1, 0]
        
        target_data = {}
        for key, data in target_metrics:
            if 'TargetType.' in data['metric_name']:
                target_type = data['metric_name'].split('.')[1]
                total_invocations = sum(dp.get('Sum', 0) for dp in data['datapoints'])
                target_data[target_type] = total_invocations
        
        if target_data and any(target_data.values()):
            # Horizontal bar chart for target types
            y_pos = range(len(target_data))
            colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
            bars = ax.barh(y_pos, list(target_data.values()), 
                          color=colors[:len(target_data)])
            ax.set_yticks(y_pos)
            ax.set_yticklabels(list(target_data.keys()))
            ax.set_title('🎯 Target Type Usage', fontweight='bold')
            ax.set_xlabel('Total Invocations')
            
            # Add value labels on bars
            for i, (bar, value) in enumerate(zip(bars, target_data.values())):
                if value > 0:
                    ax.text(value + max(target_data.values()) * 0.01, bar.get_y() + bar.get_height()/2, 
                           f'{int(value)}', ha='left', va='center', fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'No Target Data\n🎯 Configure targets to see usage', 
                   ha='center', va='center', fontsize=12)
            ax.set_title('🎯 Target Type Usage', fontweight='bold')
    else:
        axes[1, 0].text(0.5, 0.5, 'No Target Metrics\n🎯 Use different target types', 
                       ha='center', va='center', fontsize=12)
        axes[1, 0].set_title('🎯 Target Type Usage')
    
    # Error Analysis
    if error_metrics:
        ax = axes[1, 1]
        error_counts = {}
        
        for key, data in error_metrics:
            total_errors = sum(dp.get('Sum', 0) for dp in data['datapoints'])
            if total_errors > 0:
                error_counts[data['metric_name']] = total_errors
        
        if error_counts:
            # Pie chart for error distribution
            colors = ['#ff4444', '#ff8800', '#ffaa00', '#cc6600']
            wedges, texts, autotexts = ax.pie(error_counts.values(), labels=error_counts.keys(), 
                                             autopct='%1.1f%%', colors=colors[:len(error_counts)])
            for autotext in autotexts:
                autotext.set_color('white')
                autotext.set_fontweight('bold')
            ax.set_title('⚠️ Error Distribution', fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'No Errors ✅\nGateway Running Smoothly!', 
                   ha='center', va='center', fontsize=14, color='green', fontweight='bold')
            ax.set_title('⚠️ Error Analysis', fontweight='bold')
    else:
        axes[1, 1].text(0.5, 0.5, 'No Error Data\n✅ Healthy Gateway', 
                       ha='center', va='center', fontsize=12, color='green')
        axes[1, 1].set_title('⚠️ Error Analysis')
    
    plt.tight_layout()
    plt.show()
    
    # Performance summary
    if performance_metrics:
        print(f"\n⏱️ **Performance Summary**:")
        for key, data in performance_metrics:
            avg_value = sum(dp.get('Average', 0) for dp in data['datapoints']) / len(data['datapoints'])
            metric_name = data['metric_name']
            if metric_name == 'TargetExecutionTime':
                print(f"   • Tool Execution Time: {avg_value:.1f}ms (average)")
            elif metric_name == 'Latency':
                print(f"   • Gateway Latency: {avg_value:.1f}ms (average)")
            elif metric_name == 'Duration':
                print(f"   • Total Duration: {avg_value:.1f}ms (average)")

# Create visualizations
visualize_gateway_metrics(gateway_metrics)

## Step 6: Gateway Metrics Summary Table

Let's create a comprehensive summary table of all discovered gateway metrics:

In [ ]:
def create_gateway_metrics_summary_table(metrics_data: Dict) -> None:
    """Create a comprehensive summary table of all gateway metrics"""
    
    if not metrics_data:
        print("📋 No metrics data available for summary")
        return
    
    print(f"📋 **Gateway Metrics Summary Table**")
    print(f"📊 Total metrics found: {len(metrics_data)}")
    print()
    
    # Prepare data for table
    table_data = []
    
    for key, data in sorted(metrics_data.items()):
        # Extract operation and target info from dimensions
        operation = 'N/A'
        target_type = 'N/A'
        target_name = 'N/A'
        
        for dim in data['dimensions']:
            if dim['Name'] == 'Operation':
                operation = dim['Value']
            elif dim['Name'] == 'TargetType':
                target_type = dim['Value']
            elif dim['Name'] == 'TargetName':
                target_name = dim['Value']
        
        # Determine metric category
        category = '📊 General'
        if 'Mcp' in operation:
            category = '🔄 MCP'
        elif 'Target' in operation:
            category = '🎯 Target'
        elif data['metric_name'] in ['SystemErrors', 'UserErrors', 'Throttles', 'Errors']:
            category = '⚠️ Errors'
        elif data['metric_name'] in ['TargetExecutionTime', 'Latency', 'Duration']:
            category = '⏱️ Performance'
        
        # Format metric value with appropriate unit
        value = data['latest_value']
        if data['metric_name'] in ['Latency', 'Duration', 'TargetExecutionTime']:
            formatted_value = f"{value:.1f}ms"
        else:
            formatted_value = f"{value:.0f}"
        
        table_data.append({
            'Category': category,
            'Metric': data['metric_name'],
            'Operation': operation,
            'Target Type': target_type,
            'Latest Value': formatted_value,
            'Data Points': len(data['datapoints']),
            'Description': data['description'][:60] + '...' if len(data['description']) > 60 else data['description']
        })
    
    # Create DataFrame and display
    df = pd.DataFrame(table_data)
    
    # Style the DataFrame for better display
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 60)
    
    # Color-code the HTML table
    html_table = df.to_html(index=False, classes='table table-striped', escape=False)
    display(HTML(html_table))
    
    # Category and operation summaries
    category_summary = df['Category'].value_counts()
    operation_summary = df['Operation'].value_counts()
    
    print(f"\n📊 **Metrics by Category:**")
    for category, count in category_summary.items():
        print(f"   {category}: {count} metrics")
    
    if len(operation_summary) > 1 and 'N/A' not in operation_summary.index:
        print(f"\n⚙️ **Operations Summary:**")
        for operation, count in operation_summary.items():
            if operation != 'N/A':
                icon = "🔧" if operation == 'CallToolMcp' else "🚀" if operation == 'InitializeMcp' else "📋"
                print(f"   {icon} {operation}: {count} metrics")
    
    # Performance insights
    performance_metrics = df[df['Category'] == '⏱️ Performance']
    if not performance_metrics.empty:
        print(f"\n⏱️ **Performance Insights:**")
        for _, row in performance_metrics.iterrows():
            metric_name = row['Metric']
            value = row['Latest Value']
            if metric_name == 'TargetExecutionTime':
                print(f"   • Tool execution takes {value} on average")
            elif metric_name == 'Latency':
                print(f"   • Gateway response time: {value}")
            elif metric_name == 'Duration':
                print(f"   • End-to-end processing: {value}")

# Create summary table
create_gateway_metrics_summary_table(gateway_metrics)

## Step 7: Create Comprehensive Gateway Dashboard

Let's create a comprehensive CloudWatch dashboard focused on gateway performance:

In [ ]:
def create_comprehensive_gateway_dashboard(gateway_arn: str, gateway_id: str) -> bool:
    """Create a comprehensive CloudWatch dashboard for Gateway metrics"""
    
    if not gateway_arn:
        print("⚠️ No Gateway ARN available for dashboard creation")
        return False
    
    dashboard_name = f"AgentCore-Gateway-Comprehensive-{gateway_id}"
    
    print(f"🎛️ Creating Comprehensive Gateway Dashboard")
    print(f"📊 Dashboard Name: {dashboard_name}")
    print(f"🎯 Focus: Tool Performance & MCP Operations")
    
    dashboard_body = {
        "widgets": [
            # Row 1: MCP Operations Overview
            {
                "type": "metric",
                "x": 0, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "🔄 MCP Operations Activity",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "Invocations", "Resource", gateway_arn, "Operation", "CallToolMcp", {"stat": "Sum", "label": "🔧 Tool Calls", "color": "#1f77b4"}],
                        [".", ".", ".", ".", ".", "ListToolsMcp", {"stat": "Sum", "label": "📋 Tool Discovery", "color": "#ff7f0e"}],
                        [".", ".", ".", ".", ".", "InitializeMcp", {"stat": "Sum", "label": "🚀 MCP Init", "color": "#2ca02c"}],
                        [".", ".", ".", ".", ".", "InitializedNotificationMcp", {"stat": "Sum", "label": "✅ MCP Ready", "color": "#d62728"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Operations Count"}},
                    "view": "timeSeries",
                    "stacked": False
                }
            },
            {
                "type": "metric",
                "x": 12, "y": 0, "width": 12, "height": 6,
                "properties": {
                    "title": "⏱️ Gateway Performance Analysis",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "TargetExecutionTime", "Resource", gateway_arn, {"stat": "Average", "label": "🎯 Tool Execution", "yAxis": "left", "color": "#ff4444"}],
                        [".", "Latency", ".", ".", "Operation", "CallToolMcp", {"stat": "Average", "label": "🚀 Gateway Latency", "yAxis": "left", "color": "#4444ff"}],
                        [".", "Duration", ".", gateway_arn, {"stat": "Average", "label": "⏲️ Total Duration", "yAxis": "left", "color": "#44ff44"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Time (milliseconds)"}},
                    "view": "timeSeries"
                }
            },
            # Row 2: Target Types and Errors
            {
                "type": "metric",
                "x": 0, "y": 6, "width": 8, "height": 6,
                "properties": {
                    "title": "🎯 Target Type Usage",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "TargetType.LAMBDA", "Resource", gateway_arn, {"stat": "Sum", "label": "🔹 Lambda Functions", "color": "#ff9999"}],
                        [".", "TargetType.OpenAPI", ".", ".", {"stat": "Sum", "label": "🔗 OpenAPI Calls", "color": "#66b3ff"}],
                        [".", "TargetType.MCP", ".", ".", {"stat": "Sum", "label": "🔄 MCP Tools", "color": "#99ff99"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Target Invocations"}},
                    "view": "timeSeries"
                }
            },
            {
                "type": "metric",
                "x": 8, "y": 6, "width": 8, "height": 6,
                "properties": {
                    "title": "⚠️ Gateway Error Monitoring",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "SystemErrors", "Resource", gateway_arn, {"stat": "Sum", "color": "#d62728", "label": "🔴 System Errors"}],
                        [".", "UserErrors", ".", ".", {"stat": "Sum", "color": "#ff7f0e", "label": "🟠 User Errors"}],
                        [".", "Throttles", ".", ".", {"stat": "Sum", "color": "#9467bd", "label": "🟣 Throttles"}],
                        [".", "Errors", ".", ".", {"stat": "Sum", "color": "#8c564b", "label": "🔺 Total Errors"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Error Count"}},
                    "view": "timeSeries"
                }
            },
            {
                "type": "metric",
                "x": 16, "y": 6, "width": 8, "height": 6,
                "properties": {
                    "title": "📊 Current Gateway Health",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "Invocations", "Resource", gateway_arn, "Operation", "CallToolMcp", {"stat": "Sum", "label": "Tool Calls/Hour"}]
                    ],
                    "period": 3600,
                    "region": region,
                    "view": "singleValue",
                    "sparkline": True
                }
            },
            # Row 3: Detailed Analysis
            {
                "type": "metric",
                "x": 0, "y": 12, "width": 12, "height": 6,
                "properties": {
                    "title": "🔍 Tool Call Latency Distribution",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "Latency", "Resource", gateway_arn, "Operation", "CallToolMcp", {"stat": "Average", "label": "Average"}],
                        [".", ".", ".", ".", ".", ".", {"stat": "Maximum", "label": "Maximum"}],
                        [".", ".", ".", ".", ".", ".", {"stat": "Minimum", "label": "Minimum"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Latency (ms)"}},
                    "view": "timeSeries"
                }
            },
            {
                "type": "metric",
                "x": 12, "y": 12, "width": 12, "height": 6,
                "properties": {
                    "title": "🎯 Gateway vs Tool Performance Ratio",
                    "metrics": [
                        ["AWS/Bedrock-AgentCore", "TargetExecutionTime", "Resource", gateway_arn, {"stat": "Average", "label": "Tool Time", "color": "#ff6b6b"}],
                        [".", "Latency", ".", ".", "Operation", "CallToolMcp", {"stat": "Average", "label": "Gateway Time", "color": "#4ecdc4"}]
                    ],
                    "period": 300,
                    "region": region,
                    "yAxis": {"left": {"label": "Time (ms)"}},
                    "view": "timeSeries",
                    "annotations": {
                        "horizontal": [
                            {
                                "label": "Target: Tool Time > Gateway Time",
                                "value": 0
                            }
                        ]
                    }
                }
            }
        ]
    }
    
    try:
        response = cloudwatch.put_dashboard(
            DashboardName=dashboard_name,
            DashboardBody=json.dumps(dashboard_body)
        )
        
        dashboard_url = f"https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}"
        print(f"✅ Gateway dashboard created successfully!")
        print(f"📊 **View Dashboard**: {dashboard_url}")
        print()
        print(f"**Dashboard Features:**")
        print(f"   🔄 **MCP Operations**: Monitor tool calls and initialization")
        print(f"   ⏱️ **Performance Analysis**: Track gateway vs tool execution time")
        print(f"   🎯 **Target Usage**: See which target types are used most")
        print(f"   ⚠️ **Error Monitoring**: Real-time error tracking and alerting")
        print(f"   🔍 **Latency Distribution**: Min/max/average latency analysis")
        print(f"   📊 **Health Overview**: Current gateway status at a glance")
        return True
        
    except Exception as e:
        print(f"❌ Error creating dashboard: {str(e)}")
        return False

# Create comprehensive dashboard
if gateway_resource:
    create_comprehensive_gateway_dashboard(gateway_resource['gateway_arn'], gateway_resource['gateway_id'])

## Step 8: Gateway-Specific Troubleshooting Guide

Gateway metrics have unique troubleshooting considerations focused on tool performance:

In [ ]:
def gateway_metrics_troubleshooting() -> None:
    """Provide comprehensive troubleshooting guidance for Gateway metrics"""
    
    print(f"🔧 **Gateway Metrics Troubleshooting Guide**")
    print("-" * 70)
    
    print(f"### ❌ **Problem**: Not seeing CallToolMcp metrics")
    print(f"**Solution**:")
    print(f"   1. ✅ **Verify Tool Usage**: Ensure your agent is actually calling tools")
    print(f"   2. 🔧 **Check Gateway Configuration**: Verify gateway is properly configured for your agent")
    print(f"   3. 📊 **Correct Dimensions**: Use Resource + Operation = CallToolMcp")
    print(f"   4. ⏰ **Wait for Data**: Tool metrics appear 2-5 minutes after tool calls")
    print()
    
    print(f"### ❌ **Problem**: High TargetExecutionTime values")
    print(f"**Root Causes & Solutions**:")
    print(f"   1. **🔹 Lambda Function Performance**:")
    print(f"      - Cold starts causing delays")
      print(f"      - Solution: Optimize Lambda memory, use provisioned concurrency")
    print(f"   2. **🔗 External API Latency**:")
    print(f"      - OpenAPI targets responding slowly")
    print(f"      - Solution: Review external service performance, add timeouts")
    print(f"   3. **📡 Network Issues**:")
    print(f"      - Network latency between gateway and targets")
    print(f"      - Solution: Ensure targets are in same region, check security groups")
    print()
    
    print(f"### ❌ **Problem**: Gateway Latency > Target Execution Time")
    print(f"**This indicates gateway overhead issues**:")
    print(f"   1. **🔄 MCP Protocol Overhead**: Normal for complex tools")
    print(f"   2. **📊 Data Serialization**: Large payloads slow down processing")
    print(f"   3. **🚀 Gateway Resource Constraints**: High load on gateway")
    print(f"   **Solutions**:")
    print(f"     • Optimize tool input/output data sizes")
    print(f"     • Review concurrent tool usage patterns")
    print(f"     • Consider breaking large tools into smaller ones")
    print()
    
    print(f"### ❌ **Problem**: High UserErrors (4xx) from gateway")
    print(f"**Common Causes**:")
    print(f"   1. **🔧 Tool Configuration Issues**:")
    print(f"      - Incorrect tool parameters or schemas")
    print(f"      - Solution: Validate tool definitions and parameter types")
    print(f"   2. **🎯 Target Not Found**:")
    print(f"      - Lambda function doesn't exist or wrong name")
    print(f"      - Solution: Verify target configuration in gateway")
    print(f"   3. **🔐 Permission Issues**:")
    print(f"      - Gateway can't invoke Lambda functions")
    print(f"      - Solution: Check IAM roles and policies")
    print()
    
    print(f"### ❌ **Problem**: No MCP operations showing up")
    print(f"**Diagnostic Steps**:")
    print(f"   1. **🔍 Check Agent Configuration**:")
    print(f"      - Verify agent is configured to use the gateway")
    print(f"      - Ensure gateway ARN is correct in agent configuration")
    print(f"   2. **🚀 Verify Gateway Initialization**:")
    print(f"      - Look for InitializeMcp and InitializedNotificationMcp operations")
    print(f"      - If missing, gateway isn't properly connecting to agent")
    print(f"   3. **📋 Check Tool Discovery**:")
    print(f"      - ListToolsMcp should appear when agent starts")
    print(f"      - Missing indicates MCP protocol issues")
    print()
    
    print(f"### ✅ **Best Practices for Gateway Monitoring**")
    print(f"   1. **🎯 Monitor Key Ratios**:")
    print(f"      - TargetExecutionTime vs Latency (should be Target > Gateway)")
    print(f"      - Success rate: (Invocations - Errors) / Invocations")
    print()
    print(f"   2. **📊 Set Up Alarms**:")
    print(f"      - High TargetExecutionTime (> 5000ms for Lambda)")
    print(f"      - High error rates (> 5% UserErrors or any SystemErrors)")
    print(f"      - Low CallToolMcp invocations (tools not being used)")
    print()
    print(f"   3. **🔧 Performance Optimization**:")
    print(f"      - Keep tool payloads < 100KB when possible")
    print(f"      - Optimize Lambda function cold start times")
    print(f"      - Use appropriate timeout settings")
    print()
    print(f"   4. **📈 Capacity Planning**:")
    print(f"      - Monitor Invocations trends for scaling")
    print(f"      - Track different TargetType usage patterns")
    print(f"      - Plan for peak tool usage periods")
    print()
    
    print(f"### 📊 **Useful CloudWatch Queries for Gateway**")
    print(f"```")
    print(f"# Tool execution performance")
    print(f"AWS/Bedrock-AgentCore TargetExecutionTime Resource Operation CallToolMcp")
    print()
    print(f"# Gateway overhead analysis")
    print(f"AWS/Bedrock-AgentCore Latency Duration TargetExecutionTime Resource")
    print()
    print(f"# MCP protocol health")
    print(f"AWS/Bedrock-AgentCore Invocations Operation InitializeMcp CallToolMcp ListToolsMcp")
    print()
    print(f"# Target type performance comparison")
    print(f"AWS/Bedrock-AgentCore TargetType.LAMBDA TargetType.OpenAPI TargetType.MCP")
    print(f"```")
    print()
    
    print(f"### 🚨 **Critical Alerts to Set Up**")
    print(f"   1. **Tool Execution Time Alert**:")
    print(f"      - Metric: TargetExecutionTime > 10 seconds")
    print(f"      - Action: Investigate Lambda performance or external API issues")
    print()
    print(f"   2. **Gateway Error Rate Alert**:")
    print(f"      - Metric: (UserErrors + SystemErrors) / Invocations > 0.05")
    print(f"      - Action: Review tool configurations and permissions")
    print()
    print(f"   3. **MCP Connection Health Alert**:")
    print(f"      - Metric: No InitializeMcp operations in last 15 minutes")
    print(f"      - Action: Check gateway connectivity and MCP protocol")
    print()
    
    print(f"### 🆘 **Still Having Issues?**")
    print(f"   1. **📊 Compare with Memory metrics**: Cross-reference with memory operations")
    print(f"   2. **🔧 Test individual tools**: Isolate problematic tools")
    print(f"   3. **📋 Check Lambda logs**: Review CloudWatch Logs for target functions")
    print(f"   4. **🎯 Verify target types**: Ensure correct target configuration")
    print(f"   5. **🔐 Review IAM permissions**: Gateway needs proper execution role")

# Display troubleshooting guide
gateway_metrics_troubleshooting()

## 🎉 Gateway Metrics Summary

### What We Accomplished

✅ **Comprehensive Gateway Metrics Analysis**
- Discovered all MCP operations and target types
- Queried with proper dimension combinations
- Analyzed tool performance vs gateway overhead

✅ **CloudWatch Console Navigation**
- Gateway-specific console access instructions
- MCP operation monitoring guidance
- Target type performance analysis

✅ **Performance Monitoring Setup**
- Tool execution time tracking (TargetExecutionTime)
- Gateway processing time (Latency, Duration)
- MCP operation health monitoring
- Target type usage analysis

### Key Gateway Operations Monitored

| Operation | Purpose | Key Metrics | Performance Indicator |
|-----------|---------|-------------|----------------------|
| **CallToolMcp** | 🔧 Execute tools | Invocations, Latency, TargetExecutionTime | Most critical for tool performance |
| **InitializeMcp** | 🚀 Setup MCP connection | Invocations, SystemErrors | Gateway startup health |
| **InitializedNotificationMcp** | ✅ MCP ready signal | Invocations | Connection establishment success |
| **ListToolsMcp** | 📋 Discover available tools | Invocations, Latency | Tool discovery efficiency |

### Performance Analysis Framework

**🎯 Ideal Performance Pattern:**
```
TargetExecutionTime > Latency
```
This indicates tools are doing the work, not the gateway.

**⚠️ Performance Issues to Watch:**
- High TargetExecutionTime (> 5 seconds): Optimize Lambda functions
- High Latency with low TargetExecutionTime: Gateway bottleneck
- High UserErrors: Tool configuration issues
- Missing MCP operations: Connection problems

### Target Type Optimization

| Target Type | Best For | Performance Tips |
|-------------|----------|------------------|
| **LAMBDA** | 🔹 Compute-intensive tasks | Use provisioned concurrency for predictable performance |
| **OpenAPI** | 🔗 External service calls | Implement timeouts and retry logic |
| **MCP** | 🔄 Specialized agent tools | Keep payloads small, optimize protocol handling |

### Next Steps

1. **📊 Monitor Regularly**: Check dashboard for tool usage patterns
2. **🚨 Set Up Alerts**: Create alarms for high execution times and errors
3. **🎯 Optimize Tools**: Use metrics to identify slow Lambda functions
4. **📈 Scale Planning**: Monitor invocation trends for capacity planning
5. **🔧 Tool Optimization**: Review tools with high TargetExecutionTime

### 📱 Quick Access Links

- **CloudWatch Console**: Use the URLs generated in Step 4
- **Gateway Dashboard**: View the comprehensive dashboard from Step 7
- **MCP Operations**: Filter by Operation dimension in CloudWatch
- **Performance Analysis**: Compare TargetExecutionTime vs Latency

---

**🎯 You now have complete visibility into your AgentCore Gateway tool performance!**

### 🔄 Cross-Reference with Memory Metrics

For complete AgentCore monitoring, combine these Gateway metrics with the Memory metrics from the companion notebook:
- Memory operations trigger tool calls
- Tool calls create new memory events
- Combined monitoring provides full agent lifecycle visibility